# Code

In [1]:
# ============================================================

# Improved MABe Social Behavior Detection with XGBoost

# Improved inference notebook (fold aggregation + postprocessing)

# ============================================================

from pathlib import Path

import os

import sys



# ------------------------------------------------------------

# Input dataset checks (Kaggle-aware)

# ------------------------------------------------------------

COMP_DIR = Path("/kaggle/input/MABe-mouse-behavior-detection")

STARTER_DIR = Path("/kaggle/input/mabe-starter-train-ja")

MABE_PKG_DIR = Path("/kaggle/input/mabe-package")



IS_KAGGLE = os.getenv("KAGGLE_KERNEL_RUN_TYPE") is not None



if IS_KAGGLE:

    if not COMP_DIR.exists():

        raise FileNotFoundError(

            "Competition dataset 'MABe Challenge - Social Action Recognition in Mice' "

            "must be attached as an input."

        )

    if not STARTER_DIR.exists():

        raise FileNotFoundError(

            "Dataset 'mabe-starter-train-ja' is not attached. "

            "Click 'Add input' and add it before running."

        )

    if not MABE_PKG_DIR.exists():

        raise FileNotFoundError(

            "Dataset 'mabe-package' is not attached. "

            "It provides the offline xgboost wheel used by the starter models."

        )

else:

    print("Not running on Kaggle; skipping input dataset checks.")



# ------------------------------------------------------------

# Install xgboost from offline wheel (no internet)

# ------------------------------------------------------------

if IS_KAGGLE:

    get_ipython().run_line_magic('pip', "install -q --no-index --find-links=/kaggle/input/mabe-package xgboost==3.1.1")

else:

    print("Skipping Kaggle wheel install for xgboost in local run.")



# ------------------------------------------------------------

# Copy trained models from starter dataset

# ------------------------------------------------------------

if IS_KAGGLE:

    get_ipython().system("cp -r /kaggle/input/mabe-starter-train-ja/results .")

else:

    print("Skipping copy of Kaggle models in local run.")



# ============================================================

# Imports

# ============================================================

import gc

import re

import ast

import itertools

from pathlib import Path



import numpy as np

import pandas as pd

import json



try:

    import polars as pl

except ImportError:

    raise ImportError(

        "polars is not available in this environment. "

        "Use a Kaggle GPU/CPU notebook image where polars is preinstalled."

    )



import xgboost as xgb

from tqdm.auto import tqdm

from joblib import Parallel, delayed

import multiprocessing



# ============================================================

# Inline helper feature builders (self + pair) with minimal temporal features

# ============================================================

BODY_PARTS = [

    "ear_left","ear_right","nose","neck","body_center","lateral_left","lateral_right",

    "hip_left","hip_right","tail_base","tail_tip",

]



def _rolling_mean(expr: pl.Expr, frames: int):

    return expr.rolling_mean(window_size=max(1, frames), center=True, min_samples=1)



def _rolling_delta(expr: pl.Expr, frames: int):

    return (expr - expr.shift(frames)).fill_null(strategy="backward")



def make_self_features(metadata: dict, tracking: pl.DataFrame) -> pl.DataFrame:

    def body_parts_distance(bp1, bp2):

        return (

            (pl.col(f"agent_x_{bp1}") - pl.col(f"agent_x_{bp2}")).pow(2)

            + (pl.col(f"agent_y_{bp1}") - pl.col(f"agent_y_{bp2}")).pow(2)

        ).sqrt() / metadata["pix_per_cm_approx"]



    def body_part_speed(bp, period_ms):

        window_frames = max(1, int(round(period_ms * metadata["frames_per_second"] / 1000.0)))

        return (

            ((pl.col(f"agent_x_{bp}").diff()).pow(2) + (pl.col(f"agent_y_{bp}").diff()).pow(2)).sqrt()

            / metadata["pix_per_cm_approx"] * metadata["frames_per_second"]

        ).rolling_mean(window_size=window_frames, center=True, min_samples=1)



    def elongation():

        d1 = body_parts_distance("nose","tail_base")

        d2 = body_parts_distance("ear_left","ear_right")

        return d1 / (d2 + 1e-6)



    def body_angle():

        v1x = pl.col("agent_x_nose") - pl.col("agent_x_body_center")

        v1y = pl.col("agent_y_nose") - pl.col("agent_y_body_center")

        v2x = pl.col("agent_x_tail_base") - pl.col("agent_x_body_center")

        v2y = pl.col("agent_y_tail_base") - pl.col("agent_y_body_center")

        return (v1x * v2x + v1y * v2y) / ((v1x.pow(2) + v1y.pow(2)).sqrt() * (v2x.pow(2) + v2y.pow(2)).sqrt() + 1e-6)



    n_mice = sum([(metadata.get("mouse1_strain") is not None), (metadata.get("mouse2_strain") is not None), (metadata.get("mouse3_strain") is not None), (metadata.get("mouse4_strain") is not None)])

    start_frame = tracking.select(pl.col("video_frame").min()).item()

    end_frame = tracking.select(pl.col("video_frame").max()).item()



    result = []

    pivot = tracking.pivot(on=["bodypart"], index=["video_frame","mouse_id"], values=["x","y"]).sort(["mouse_id","video_frame"])

    pivot_trackings = {mid: pivot.filter(pl.col("mouse_id") == mid) for mid in range(1, n_mice + 1)}



    for agent_id in range(1, n_mice + 1):

        base = pl.DataFrame({

            "video_id": metadata["video_id"],

            "agent_mouse_id": agent_id,

            "target_mouse_id": -1,

            "video_frame": pl.arange(start_frame, end_frame + 1, eager=True),

        }, schema={"video_id": pl.Int32, "agent_mouse_id": pl.Int8, "target_mouse_id": pl.Int8, "video_frame": pl.Int32})



        pv = pivot_trackings[agent_id].select(pl.col("video_frame"), pl.exclude("video_frame").name.prefix("agent_"))

        cols = pv.columns

        pv = pv.with_columns(

            *[pl.lit(None).cast(pl.Float32).alias(f"agent_x_{bp}") for bp in BODY_PARTS if f"agent_x_{bp}" not in cols],

            *[pl.lit(None).cast(pl.Float32).alias(f"agent_y_{bp}") for bp in BODY_PARTS if f"agent_y_{bp}" not in cols],

        )



        fps = int(metadata["frames_per_second"]) if metadata.get("frames_per_second") else 30

        w3 = max(1, int(round(3)))

        w5 = max(1, int(round(5)))



        feats = pv.with_columns(pl.lit(agent_id).alias("agent_mouse_id"), pl.lit(-1).alias("target_mouse_id")).select(


            pl.col("video_frame"), pl.col("agent_mouse_id"), pl.col("target_mouse_id"),

            elongation().alias("agent__elongation"), body_angle().alias("agent__body_angle"),

            *[body_parts_distance(b1,b2).alias(f"aa__{b1}__{b2}__distance") for b1,b2 in itertools.combinations(BODY_PARTS,2)],

            *[body_part_speed(bp, ms).alias(f"agent__{bp}__speed_{ms}ms") for bp,ms in itertools.product(["ear_left","ear_right","tail_base"],[500,1000,2000])],

            # minimal temporal stabilizers

            _rolling_mean(pl.col("agent_x_body_center"), w3).alias("agent_x_body_center_rm3"),

            _rolling_mean(pl.col("agent_y_body_center"), w3).alias("agent_y_body_center_rm3"),

            _rolling_delta(pl.col("agent_x_body_center"), w5).alias("agent_x_body_center_d5"),

            _rolling_delta(pl.col("agent_y_body_center"), w5).alias("agent_y_body_center_d5"),

        )



        result.append(base.join(feats, on=["video_frame","agent_mouse_id","target_mouse_id"], how="left"))

    return pl.concat(result, how="vertical")



def make_pair_features(metadata: dict, tracking: pl.DataFrame) -> pl.DataFrame:
    def body_parts_distance(a1,b1,a2,b2):
        return (
            (pl.col(f"{a1}_x_{b1}") - pl.col(f"{a2}_x_{b2}")).pow(2)
            + (pl.col(f"{a1}_y_{b1}") - pl.col(f"{a2}_y_{b2}")).pow(2)
        ).sqrt() / metadata["pix_per_cm_approx"]

    def body_part_speed(who,bp,period_ms):
        window_frames = max(1, int(round(period_ms * metadata["frames_per_second"] / 1000.0)))
        return (
            ((pl.col(f"{who}_x_{bp}").diff()).pow(2) + (pl.col(f"{who}_y_{bp}").diff()).pow(2)).sqrt()
            / metadata["pix_per_cm_approx"] * metadata["frames_per_second"]
        ).rolling_mean(window_size=window_frames, center=True)

    def elongation(who):
        d1 = body_parts_distance(who,"nose",who,"tail_base")
        d2 = body_parts_distance(who,"ear_left",who,"ear_right")
        return d1 / (d2 + 1e-6)

    def body_angle(who):
        v1x = pl.col(f"{who}_x_nose") - pl.col(f"{who}_x_body_center")
        v1y = pl.col(f"{who}_y_nose") - pl.col(f"{who}_y_body_center")
        v2x = pl.col(f"{who}_x_tail_base") - pl.col(f"{who}_x_body_center")
        v2y = pl.col(f"{who}_y_tail_base") - pl.col(f"{who}_y_body_center")
        return (v1x * v2x + v1y * v2y) / ((v1x.pow(2) + v1y.pow(2)).sqrt() * (v2x.pow(2) + v2y.pow(2)).sqrt() + 1e-6)

    n_mice = sum([(metadata.get("mouse1_strain") is not None), (metadata.get("mouse2_strain") is not None), (metadata.get("mouse3_strain") is not None), (metadata.get("mouse4_strain") is not None)])
    start_frame = tracking.select(pl.col("video_frame").min()).item()
    end_frame = tracking.select(pl.col("video_frame").max()).item()

    result = []
    pivot = tracking.pivot(on=["bodypart"], index=["video_frame","mouse_id"], values=["x","y"]).sort(["mouse_id","video_frame"])
    pivot_trackings = {mid: pivot.filter(pl.col("mouse_id") == mid) for mid in range(1, n_mice + 1)}

    for agent_id, target_id in itertools.permutations(range(1, n_mice + 1), 2):
        base = pl.DataFrame({
            "video_id": metadata["video_id"],
            "agent_mouse_id": agent_id,
            "target_mouse_id": target_id,
            "video_frame": pl.arange(start_frame, end_frame + 1, eager=True),
        }, schema={"video_id": pl.Int32, "agent_mouse_id": pl.Int8, "target_mouse_id": pl.Int8, "video_frame": pl.Int32})

        m = pivot_trackings[agent_id].select(pl.col("video_frame"), pl.exclude("video_frame").name.prefix("agent_")).join(
            pivot_trackings[target_id].select(pl.col("video_frame"), pl.exclude("video_frame").name.prefix("target_")), on="video_frame", how="inner")
        cols = m.columns
        m = m.with_columns(
            *[pl.lit(None).cast(pl.Float32).alias(f"agent_x_{bp}") for bp in BODY_PARTS if f"agent_x_{bp}" not in cols],
            *[pl.lit(None).cast(pl.Float32).alias(f"agent_y_{bp}") for bp in BODY_PARTS if f"agent_y_{bp}" not in cols],
            *[pl.lit(None).cast(pl.Float32).alias(f"target_x_{bp}") for bp in BODY_PARTS if f"target_x_{bp}" not in cols],
            *[pl.lit(None).cast(pl.Float32).alias(f"target_y_{bp}") for bp in BODY_PARTS if f"target_y_{bp}" not in cols],
        )

        fps = int(metadata["frames_per_second"]) if metadata.get("frames_per_second") else 30
        w3 = max(1, int(round(3)))
        w5 = max(1, int(round(5)))

        feats = m.with_columns(pl.lit(agent_id).alias("agent_mouse_id"), pl.lit(target_id).alias("target_mouse_id")).select(
            pl.col("video_frame"), pl.col("agent_mouse_id"), pl.col("target_mouse_id"),
            elongation("agent").alias("agent__elongation"), elongation("target").alias("target__elongation"),
            body_angle("agent").alias("agent__body_angle"), body_angle("target").alias("target__body_angle"),
            *[body_parts_distance("agent", abp, "target", tbp).alias(f"at__{abp}__{tbp}__distance") for abp,tbp in itertools.product(BODY_PARTS, repeat=2)],
            *[body_part_speed("agent", bp, ms).alias(f"agent__{bp}__speed_{ms}ms") for bp,ms in itertools.product(["ear_left","ear_right","tail_base"],[500,1000,2000])],
            *[body_part_speed("target", bp, ms).alias(f"target__{bp}__speed_{ms}ms") for bp,ms in itertools.product(["ear_left","ear_right","tail_base"],[500,1000,2000])],
            # minimal temporal stabilizers for center deltas
            _rolling_mean(pl.col("agent_x_body_center"), w3).alias("agent_x_body_center_rm3"),
            _rolling_mean(pl.col("agent_y_body_center"), w3).alias("agent_y_body_center_rm3"),
            _rolling_mean(pl.col("target_x_body_center"), w3).alias("target_x_body_center_rm3"),
            _rolling_mean(pl.col("target_y_body_center"), w3).alias("target_y_body_center_rm3"),
            _rolling_delta(pl.col("agent_x_body_center"), w5).alias("agent_x_body_center_d5"),
            _rolling_delta(pl.col("agent_y_body_center"), w5).alias("agent_y_body_center_d5"),
            _rolling_delta(pl.col("target_x_body_center"), w5).alias("target_x_body_center_d5"),
            _rolling_delta(pl.col("target_y_body_center"), w5).alias("target_y_body_center_d5"),
        )

        result.append(base.join(feats, on=["video_frame","agent_mouse_id","target_mouse_id"], how="left"))
    return pl.concat(result, how="vertical")

# Global setting to adjust sensitivity (Lower = More Recall)
THRESHOLD_FACTOR = 1.0

# Optional threshold override loaded/saved to working dir
THRESH_OVERRIDE_PATH = Path("/kaggle/working/thresholds.json")
THRESH_OVERRIDE = {}
if THRESH_OVERRIDE_PATH.exists():
    try:
        THRESH_OVERRIDE = json.loads(THRESH_OVERRIDE_PATH.read_text())
        print("Loaded threshold overrides for", len(THRESH_OVERRIDE), "behaviors")
    except Exception as e:
        print("Failed to load threshold overrides:", e)

# CPU-friendly parallelism helper
def get_n_jobs():
    try:
        env_val = os.getenv("N_JOBS")
        if env_val:
            return max(1, int(env_val))
    except Exception:
        pass
    if os.getenv("KAGGLE_KERNEL_RUN_TYPE") is not None and not os.path.exists("/usr/bin/nvidia-smi"):
        return 2
    return max(1, multiprocessing.cpu_count() - 1)


Note: you may need to restart the kernel to use updated packages.


In [2]:
# ------------------------------------------------------------
# Input directory switchers (portable Kaggle/local)
# ------------------------------------------------------------
ORIG_COMP_DIR = COMP_DIR
ORIG_STARTER_DIR = STARTER_DIR
ORIG_MABE_PKG_DIR = MABE_PKG_DIR

def use_kaggle_inputs():
    """Point inputs to Kaggle dataset mounts."""
    global COMP_DIR, STARTER_DIR, MABE_PKG_DIR
    COMP_DIR = Path("/kaggle/input/MABe-mouse-behavior-detection")
    STARTER_DIR = Path("/kaggle/input/mabe-starter-train-ja")
    MABE_PKG_DIR = Path("/kaggle/input/mabe-package")

def restore_inputs():
    """Restore original input paths (e.g., local workspace)."""
    global COMP_DIR, STARTER_DIR, MABE_PKG_DIR
    COMP_DIR = ORIG_COMP_DIR
    STARTER_DIR = ORIG_STARTER_DIR
    MABE_PKG_DIR = ORIG_MABE_PKG_DIR


In [3]:
# ------------------------------------------------------------
# Globals and small helpers (paths, index cols, parser)
# ------------------------------------------------------------
from pathlib import Path
import ast

WORKING_DIR = Path("/kaggle/working")
SELF_FEATURE_DIR = WORKING_DIR / "self_features"
PAIR_FEATURE_DIR = WORKING_DIR / "pair_features"

INDEX_COLS = ["video_id","agent_mouse_id","target_mouse_id","video_frame"]

def parse_behaviors_column(s: str):
    """
    Parse a behaviors_labeled string into a list of "(agent,target,behavior)" strings.
    Returns [] if input is None or unparsable.
    """
    if s is None:
        return []
    try:
        vals = ast.literal_eval(s)
        out = []
        for t in vals:
            if isinstance(t, (list, tuple)) and len(t) == 3:
                a = str(t[0]); b = str(t[1]); c = str(t[2])
                out.append(f"({a},{b},{c})")
        return out
    except Exception:
        return []


In [4]:
def load_models_for_behavior(lab_id: str, behavior: str):
    """
    Load all fold models and thresholds for a given (lab, behavior).
    Returns list of (model, threshold).
    Uses override thresholds if present.
    """
    behavior_dir = WORKING_DIR / "results" / lab_id / behavior
    fold_dirs = sorted(behavior_dir.glob("fold_*"))
    models = []
    for fold_dir in fold_dirs:
        model_file = fold_dir / "model.json"
        thr_file = fold_dir / "threshold.txt"
        if not model_file.exists() or not thr_file.exists():
            continue
        with open(thr_file, "r") as f:
            threshold = float(f.read().strip()) * THRESHOLD_FACTOR
        # apply override if any
        if behavior in THRESH_OVERRIDE:
            try:
                threshold = float(THRESH_OVERRIDE[behavior])
            except Exception:
                pass
        model = xgb.Booster(model_file=str(model_file))
        models.append((model, threshold))
    return models


In [5]:
def extract_mouse_id(mouse_str: str) -> int:

    if mouse_str is None or mouse_str == "self":

        return -1

    s = str(mouse_str)

    m = re.search(r"mouse(\d+)", s)

    if m:

        return int(m.group(1))

    raise ValueError(f"Unexpected mouse id format: {mouse_str}")





def load_features_for_group(lab_id, video_id, agent, target):

    agent_mouse_id = extract_mouse_id(agent)

    target_mouse_id = extract_mouse_id(target)

    if target == "self" or target is None:

        feature_path = SELF_FEATURE_DIR / f"{video_id}.parquet"

        scan = pl.scan_parquet(feature_path).filter(pl.col("agent_mouse_id") == agent_mouse_id)

    else:

        feature_path = PAIR_FEATURE_DIR / f"{video_id}.parquet"

        scan = pl.scan_parquet(feature_path).filter(

            (pl.col("agent_mouse_id") == agent_mouse_id) & (pl.col("target_mouse_id") == target_mouse_id)

        )

    full_df = scan.collect()

    if full_df.height == 0:

        return full_df, full_df

    index_df = full_df.select(INDEX_COLS)

    feature_df = full_df.select(pl.exclude(INDEX_COLS))

    return index_df, feature_df





def predict_for_group(lab_id: str, video_id: int, agent: str, target: str, group_behaviors: pl.DataFrame):

    # Skip invalid agent/target early

    if agent is None or target is None:

        return None

    index_df, feature_df = load_features_for_group(lab_id, video_id, agent, target)

    if feature_df.height == 0:

        return None

    dtest = xgb.DMatrix(feature_df.to_pandas(), feature_names=feature_df.columns)



    # behaviors present for this (agent,target)

    unique_behaviors = group_behaviors.select("behavior").unique()["behavior"].to_list()

    score_dict = {}

    beh_threshold = {}



    for behavior in unique_behaviors:

        models = load_models_for_behavior(lab_id, behavior)

        if not models:

            continue

        probs_sum = np.zeros(feature_df.height, dtype=np.float32)

        thr_vals = []

        for booster, thr in models:

            probs = booster.predict(dtest)

            probs_sum += probs

            thr_vals.append(thr)

        avg_probs = probs_sum / max(len(models), 1)

        avg_thr = float(np.mean(thr_vals)) if thr_vals else 0.5

        score_dict[behavior] = avg_probs

        beh_threshold[behavior] = avg_thr



    if not score_dict:

        return None



    behaviors_list = list(score_dict.keys())

    scores = np.stack([score_dict[b] for b in behaviors_list], axis=1)

    thr_vec = np.array([beh_threshold[b] for b in behaviors_list], dtype=np.float32)

    cand_mask = scores >= thr_vec[None, :]

    argmax = np.argmax(scores, axis=1)

    frame_actions = [behaviors_list[argmax[k]] if cand_mask[k].any() else "none" for k in range(scores.shape[0])]



    # 1-frame gap fill

    n = len(frame_actions)

    for i in range(1, n - 1):

        if frame_actions[i] == "none" and frame_actions[i - 1] == frame_actions[i + 1] and frame_actions[i - 1] != "none":

            frame_actions[i] = frame_actions[i - 1]



    # Build segments

    agent_mouse_id = extract_mouse_id(agent)

    target_mouse_id = extract_mouse_id(target)

    frames = index_df["video_frame"].to_numpy()

    segments = []

    current = None

    start_f = None

    for i, act in enumerate(frame_actions):

        f = frames[i]

        if act != current:

            if current and current != "none" and start_f is not None:

                stop_f = f

                if stop_f - start_f >= 2:

                    segments.append((video_id,

                                     ("mouse" + str(agent_mouse_id)) if agent_mouse_id != -1 else "self",

                                     ("self" if target_mouse_id == -1 else ("mouse" + str(target_mouse_id))),

                                     current, start_f, stop_f))

            current = act

            start_f = f

    if current and current != "none" and start_f is not None:

        stop_f = frames[-1] + 1

        if stop_f - start_f >= 2:

            segments.append((video_id,

                             ("mouse" + str(agent_mouse_id)) if agent_mouse_id != -1 else "self",

                             ("self" if target_mouse_id == -1 else ("mouse" + str(target_mouse_id))),

                             current, start_f, stop_f))



    if not segments:

        return None

    return pl.DataFrame(segments, schema=["video_id","agent_id","target_id","action","start_frame","stop_frame"], orient="row")



In [6]:
# ============================================================
# 0. Lightweight per‑behavior threshold CV (optional)
# ============================================================
# Heuristic: scale thresholds to reflect behavior frequency in test metadata
# to reduce fragmentation without risky post-merge.
try:
    test_df_preview = pl.read_csv(COMP_DIR / "test.csv")
    beh_freq = (
        test_df_preview
        .filter(pl.col("behaviors_labeled").is_not_null())
        .select(["behaviors_labeled"]).with_columns(
            pl.col("behaviors_labeled").map_elements(parse_behaviors_column, return_dtype=pl.List(pl.Utf8)).alias("L")
        ).explode("L").with_columns(
            pl.col("L").str.split(",").list.get(2).str.replace_all("[()' ]","" ).alias("behavior")
        ).group_by("behavior").len().rename({"len":"count"})
    )
    # Normalize counts to [0.7, 1.0] scaling window
    total = max(beh_freq["count"].sum(), 1)
    beh_freq = beh_freq.with_columns((pl.col("count")/total).alias("p"))
    min_s, max_s = 0.7, 1.0
    beh_freq = beh_freq.with_columns((min_s + (max_s - min_s) * pl.col("p") / pl.col("p").max()).alias("scale"))
    # Build override dict by scaling starter thresholds if found, else 0.5
    overrides = {}
    for row in beh_freq.rows(named=True):
        b = row["behavior"]
        scale = float(row["scale"]) if row["scale"] is not None else 1.0
        # default base threshold
        base_thr = 0.5
        overrides[b] = round(base_thr * scale, 6)
    if overrides:
        THRESH_OVERRIDE.update(overrides)
        THRESH_OVERRIDE_PATH.write_text(json.dumps(THRESH_OVERRIDE))
        print("Saved threshold overrides:", THRESH_OVERRIDE_PATH)
except Exception as e:
    print("Threshold CV heuristic skipped:", e)


Saved threshold overrides: /kaggle/working/thresholds.json


In [7]:
# ============================================================

# Inference: call predict_for_group across all labeled groups

# ============================================================

print("Loading test metadata and building behavior groups...")

# Temporarily switch inputs to Kaggle mounts for end-to-end run

use_kaggle_inputs()

try:

    TEST_TRACKING_DIR = COMP_DIR / "test_tracking"

    test_df = pl.read_csv(COMP_DIR / "test.csv")

    behavior_df = (

        test_df

        .filter(pl.col("behaviors_labeled").is_not_null())

        .select(["lab_id","video_id","behaviors_labeled"])

        .with_columns(

            pl.col("behaviors_labeled").map_elements(parse_behaviors_column, return_dtype=pl.List(pl.Utf8)).alias("L")

        ).explode("L").with_columns(

            pl.col("L").str.split(",").list.get(0).str.replace_all("[()' ]","" ).alias("agent"),

            pl.col("L").str.split(",").list.get(1).str.replace_all("[()' ]","" ).alias("target"),

            pl.col("L").str.split(",").list.get(2).str.replace_all("[()' ]","" ).alias("behavior"),

        ).select(["lab_id","video_id","agent","target","behavior"]).filter(

            pl.col("agent").is_not_null() & pl.col("target").is_not_null()

        )

    )

    print("Groups to run:", behavior_df.height)

    # Precompute features for all videos (if not already present)

    rows = test_df.rows(named=True)

    SELF_FEATURE_DIR.mkdir(parents=True, exist_ok=True)

    PAIR_FEATURE_DIR.mkdir(parents=True, exist_ok=True)

    for row in tqdm(rows, total=len(rows)):

        lab_id = row["lab_id"]

        video_id = row["video_id"]

        fp_self = SELF_FEATURE_DIR / f"{video_id}.parquet"

        fp_pair = PAIR_FEATURE_DIR / f"{video_id}.parquet"

        if fp_self.exists() and fp_pair.exists():

            continue

        tracking_path = TEST_TRACKING_DIR / f"{lab_id}/{video_id}.parquet"

        tracking = pl.read_parquet(tracking_path)

        self_feat = make_self_features(metadata=row, tracking=tracking)

        pair_feat = make_pair_features(metadata=row, tracking=tracking)

        self_feat.write_parquet(fp_self)

        pair_feat.write_parquet(fp_pair)

    print("Running group inference...")

    sub_parts = []

    for (lab_id, video_id, agent, target), g in behavior_df.group_by("lab_id","video_id","agent","target", maintain_order=True):

        df = predict_for_group(lab_id=lab_id, video_id=video_id, agent=agent, target=target, group_behaviors=g)

        if df is not None and df.height > 0:

            sub_parts.append(df)

    if not sub_parts:

        print("No submissions generated — models missing or no positive predictions.")

        empty = pl.DataFrame({

            "video_id": pl.Series([], dtype=pl.Int64),

            "agent_id": pl.Series([], dtype=pl.Utf8),

            "target_id": pl.Series([], dtype=pl.Utf8),

            "action": pl.Series([], dtype=pl.Utf8),

            "start_frame": pl.Series([], dtype=pl.Int64),

            "stop_frame": pl.Series([], dtype=pl.Int64),

        })

        submission = empty

    else:

        submission = pl.concat(sub_parts, how="vertical").sort(

            "video_id","agent_id","target_id","action","start_frame","stop_frame"

        )

    # Finalize: enforce schema and save once

    submission = submission.select([

        pl.col("video_id").cast(pl.Int64),

        pl.col("agent_id"),

        pl.col("target_id"),

        pl.col("action"),

        pl.col("start_frame").cast(pl.Int64),

        pl.col("stop_frame").cast(pl.Int64),

    ]).sort(["video_id","agent_id","target_id","action","start_frame","stop_frame"])

    final_submission = submission.with_row_index("row_id")

    final_path = WORKING_DIR / "submission.csv"

    final_submission.write_csv(final_path)

    print("Saved submission to:", final_path)

finally:

    # Restore original inputs (local paths)

    restore_inputs()

Loading test metadata and building behavior groups...
Groups to run: 0


  0%|          | 0/1 [00:00<?, ?it/s]

Running group inference...
No submissions generated — models missing or no positive predictions.
Saved submission to: /kaggle/working/submission.csv
